# AksaraDetect - Full Project di Google Colab

**Sebelum mulai:**
1. Klik **Runtime** -> **Change runtime type** -> pilih **T4 GPU** -> Save
2. Ganti `GANTI_API_KEY_KAMU` di Cell 2 dengan API key Roboflow kamu
3. Klik **Runtime** -> **Run all**
4. Tunggu ~15 menit, lalu klik URL yang muncul di Cell terakhir

**Cara dapat API Key:** buka https://app.roboflow.com -> klik foto profil -> Settings -> Roboflow API

In [1]:
# Cell 1 - Install semua dependensi
!nvidia-smi
!pip install ultralytics roboflow streamlit pyngrok PyYAML matplotlib seaborn pandas Pillow scikit-learn -q
print("Install selesai!")

Mon May 11 03:50:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 - Download dataset dari Roboflow
from roboflow import Roboflow
import yaml, os

API_KEY = "cSaQdx9RY5YTrc7KjCAs"  # <-- GANTI INI
# Cara dapat: buka app.roboflow.com -> Settings -> Roboflow API

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("novalrizkiansyah-ymail-com").project("aksara-ulu-rejang")
dataset = project.version(4).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
base = os.path.dirname(DATA_YAML)

with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

data["train"] = base + "/train/images"
data["val"]   = base + "/valid/images"
data["test"]  = base + "/test/images"

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f)

print(f"Dataset siap! Kelas: {data['nc']}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Aksara-Ulu-Rejang-4 in yolov8:: 100%|██████████| 3271/3271 [00:01<00:00, 3088.60it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset siap! Kelas: 253


In [3]:
# Cell 3 - Clone project dari GitHub
!git clone https://github.com/ReffkiAndreaPratama/aksara-detect.git /content/app
print("Project berhasil di-clone!")

Cloning into '/content/app'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 38 (delta 4), reused 32 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 39.18 KiB | 13.06 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Project berhasil di-clone!


In [4]:
# Cell 4 - Training YOLOv8 dengan GPU (~10-15 menit)
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data     = DATA_YAML,
    epochs   = 50,
    imgsz    = 640,
    batch    = 32,
    device   = 0,
    project  = "/content/runs",
    name     = "aksara_ulu",
    exist_ok = True,
)
print("Training selesai!")
print("mAP50:", results.results_dict.get("metrics/mAP50(B)", "N/A"))

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Aksara-Ulu-Rejang-4/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aksara_ulu, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

In [69]:
# Cell 5 - Setup model, label map, dan config
import os, json, shutil, yaml

for d in ["/content/app/models", "/content/app/results", "/content/app/logs"]:
    os.makedirs(d, exist_ok=True)

shutil.copy("/content/runs/aksara_ulu/weights/best.pt", "/content/app/models/best.pt")

with open(DATA_YAML) as f:
    d = yaml.safe_load(f)
label_map = {str(i): name for i, name in enumerate(d["names"])}
with open("/content/app/models/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

config_lines = [
    "# config.py - Dibuat secara otomatis di Colab.\n", # Added print statement indicator
    "import os\n",
    'BASE_DIR        = "/content/app"\n',
    f'DATA_YAML       = "{DATA_YAML}"\n',
    'MODEL_DIR       = "/content/app/models"\n',
    'RESULT_DIR      = "/content/app/results"\n',
    'LOG_DIR         = "/content/app/logs"\n',
    'RUNS_DIR        = "/content/app/runs"\n',
    'MODEL_BEST_PATH = "/content/app/models/best.pt"\n',
    'MODEL_LAST_PATH = "/content/app/models/last.pt"\n',
    'LABEL_MAP_PATH  = "/content/app/models/label_map.json"\n',
    'METRICS_PATH    = "/content/app/models/metrics.json"\n',
    'PREDICT_LOG     = "/content/app/logs/prediction_log.json"\n',
    "CONF_THRESHOLD  = 0.25\n",
    "IOU_THRESHOLD   = 0.45\n",
    "IMG_SIZE        = 640\n",
    "EPOCHS          = 50\n",
    "BATCH_SIZE      = 32\n",
    "PATIENCE        = 100\n", # Default YOLOv8 patience
    "DEVICE          = '0'\n", # Default GPU 0
    "WORKERS         = 8\n",    # Default YOLOv8 workers
    'APP_NAME        = "AksaraDetect"\n',
    'APP_VERSION     = "1.0.0"\n',
    'APP_TAGLINE     = "Aksara Ulu Rejang - YOLOv8"\n',
]
with open("/content/app/src/config.py", "w") as f:
    f.writelines(config_lines)

print(f"Setup selesai! {len(label_map)} kelas siap.")


Setup selesai! 253 kelas siap.


In [67]:
print('\nReading content of /content/app/src/config.py:')
!cat /content/app/src/config.py



Reading content of /content/app/src/config.py:
import os
BASE_DIR        = "/content/app"
DATA_YAML       = "/content/Aksara-Ulu-Rejang-4/data.yaml"
MODEL_DIR       = "/content/app/models"
RESULT_DIR      = "/content/app/results"
LOG_DIR         = "/content/app/logs"
RUNS_DIR        = "/content/app/runs"
MODEL_BEST_PATH = "/content/app/models/best.pt"
MODEL_LAST_PATH = "/content/app/models/last.pt"
LABEL_MAP_PATH  = "/content/app/models/label_map.json"
METRICS_PATH    = "/content/app/models/metrics.json"
PREDICT_LOG     = "/content/app/logs/prediction_log.json"
CONF_THRESHOLD  = 0.25
IOU_THRESHOLD   = 0.45
IMG_SIZE        = 640
EPOCHS          = 50
BATCH_SIZE      = 32
PATIENCE        = 100
DEVICE          = '0'
WORKERS         = 8
APP_NAME        = "AksaraDetect"
APP_VERSION     = "1.0.0"
APP_TAGLINE     = "Aksara Ulu Rejang - YOLOv8"


In [21]:
# Cell 6 - Jalankan GUI Streamlit + buat URL publik
import subprocess, time, sys
from pyngrok import ngrok

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run",
     "/content/app/src/app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    cwd="/content/app/src"
)

time.sleep(10)

tunnel = ngrok.connect(8501)
print("=" * 55)
print("AKSARADETECT SIAP!")
print("Buka URL ini di browser:")
print("  " + tunnel.public_url)
print("=" * 55)
print("URL aktif selama session Colab berjalan")

AKSARADETECT SIAP!
Buka URL ini di browser:
  https://storage-evacuee-skeptic.ngrok-free.dev
URL aktif selama session Colab berjalan


In [17]:
# Cell 6 - Jalankan GUI Streamlit + buat URL publik

import subprocess
import time
import sys
import os

from pyngrok import ngrok

# ==============================
# PASANG AUTHTOKEN NGROK
# ==============================
ngrok.set_auth_token("3DYyag2hTLupTpR7USa67cAViGP_4y3FB2fsQcA53q4AMDopQ")

# ==============================
# JALANKAN STREAMLIT
# ==============================
log_file_path = "/content/app/src/streamlit_errors.log"
os.makedirs(os.path.dirname(log_file_path), exist_ok=True)

with open(log_file_path, "w") as log_file:
    proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "/content/app/src/app.py",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
            "--server.enableCORS",
            "false",
            "--server.enableXsrfProtection",
            "false",
        ],
        cwd="/content/app/src",
        stdout=log_file,
        stderr=log_file,
    )

# tunggu streamlit startup
time.sleep(10)

# ==============================
# BUAT PUBLIC URL
# ==============================
tunnel = ngrok.connect(8501)

print("=" * 55)
print("AKSARADETECT SIAP!")
print("Buka URL ini di browser:")
print(tunnel.public_url)
print("=" * 55)
print("URL aktif selama session Colab berjalan")

AKSARADETECT SIAP!
Buka URL ini di browser:
https://storage-evacuee-skeptic.ngrok-free.dev
URL aktif selama session Colab berjalan


### Streamlit Error Log

Di bawah ini adalah output dari log error Streamlit. Ini akan membantu kita mengetahui mengapa halaman-halaman tersebut kosong.

In [47]:
log_file_path = "/content/app/src/streamlit_errors.log"

try:
    with open(log_file_path, "r") as f:
        errors = f.read()
    if errors:
        print("Streamlit Error Log:\n")
        print(errors)
    else:
        print("Tidak ada error yang tercatat di log file.")
except FileNotFoundError:
    print(f"Log file tidak ditemukan di {log_file_path}. Pastikan Cell 6 sudah dijalankan dengan benar.")


Streamlit Error Log:



2026-05-11 04:53:02.520 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.70.45:8501




In [68]:
print('\nReading content of /content/app/src/_pages/settings.py:')
!cat /content/app/src/_pages/settings.py


Reading content of /content/app/src/_pages/settings.py:
"""pages/settings.py — Konfigurasi dan info project."""

import os, sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import streamlit as st
import config, utils
from components.ui import page_header, divider


def _size(path):
    if not os.path.isfile(path): return "—"
    s = os.path.getsize(path)
    return f"{s/1024**2:.1f} MB" if s >= 1024**2 else \
           f"{s/1024:.1f} KB"   if s >= 1024    else f"{s} B"


def render():
    page_header("Settings & Info", "Konfigurasi model dan status project")

    # ── Config ────────────────────────────────────────────────────────────────
    st.markdown("""
    <div style="font-family:'Space Grotesk',sans-serif;font-size:18px;font-weight:600;
                color:#F0F0FF;margin-bottom:16px">Konfigurasi Training</div>
    """, unsafe_allow_html=True)

    items = [
        ("Model Size",    f"YOLOv8{config.IMG_SIZE}"),
        ("Epochs",        st

In [49]:
# Fix AttributeError: module 'config' has no attribute 'MODEL_SIZE' in _pages/settings.py

file_path = "/content/app/src/_pages/settings.py"

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if "config.MODEL_SIZE" in line:
        new_lines.append(line.replace("config.MODEL_SIZE", "config.IMG_SIZE"))
    else:
        new_lines.append(line)

with open(file_path, "w") as f:
    f.writelines(new_lines)

print("Fixed 'MODEL_SIZE' to 'IMG_SIZE' in _pages/settings.py!")

# Verify the change
print('\nVerifying content of /content/app/src/_pages/settings.py:')
!cat /content/app/src/_pages/settings.py


Fixed 'MODEL_SIZE' to 'IMG_SIZE' in _pages/settings.py!

Verifying content of /content/app/src/_pages/settings.py:
"""pages/settings.py — Konfigurasi dan info project."""

import os, sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import streamlit as st
import config, utils
from components.ui import page_header, divider


def _size(path):
    if not os.path.isfile(path): return "—"
    s = os.path.getsize(path)
    return f"{s/1024**2:.1f} MB" if s >= 1024**2 else \
           f"{s/1024:.1f} KB"   if s >= 1024    else f"{s} B"


def render():
    page_header("Settings & Info", "Konfigurasi model dan status project")

    # ── Config ────────────────────────────────────────────────────────────────
    st.markdown("""
    <div style="font-family:'Space Grotesk',sans-serif;font-size:18px;font-weight:600;
                color:#F0F0FF;margin-bottom:16px">Konfigurasi Training</div>
    """, unsafe_allow_html=True)

    items = [
        ("Model Size",   

In [52]:
# Restart Streamlit after fixing settings.py
import subprocess, time, sys
from pyngrok import ngrok

# Kill proses lama
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Clear existing ngrok tunnels
for tunnel_obj in ngrok.get_tunnels():
    ngrok.disconnect(tunnel_obj.public_url)
    print(f"Closed old ngrok tunnel: {tunnel_obj.public_url}")

# Jalankan ulang Streamlit dengan `stderr` ke log file
log_file_path = "/content/app/src/streamlit_errors.log"
os.makedirs(os.path.dirname(log_file_path), exist_ok=True)

with open(log_file_path, "w") as log_file:
    proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "/content/app/src/app.py",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
            "--server.enableCORS",
            "false",
            "--server.enableXsrfProtection",
            "false",
        ],
        cwd="/content/app/src", # Jalankan dari src agar path relatif pages/_pages tetap bekerja
        stdout=log_file,
        stderr=log_file,
    )

time.sleep(10)

tunnel = ngrok.connect(8501)
print("=" * 50)
print("AKSARADETECT SIAP (Setelah perbaikan settings.py)!")
print("Buka:", tunnel.public_url)
print("=" * 50)


Closed old ngrok tunnel: https://storage-evacuee-skeptic.ngrok-free.dev
AKSARADETECT SIAP (Setelah perbaikan settings.py)!
Buka: https://storage-evacuee-skeptic.ngrok-free.dev


In [71]:
log_file_path = "/content/app/src/streamlit_errors.log"

try:
    with open(log_file_path, "r") as f:
        errors = f.read()
    if errors:
        print("Streamlit Error Log:\n")
        print(errors)
    else:
        print("Tidak ada error yang tercatat di log file.")
except FileNotFoundError:
    print(f"Log file tidak ditemukan di {log_file_path}. Pastikan Cell 6 sudah dijalankan dengan benar.")


Streamlit Error Log:



2026-05-11 05:04:49.309 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.70.45:8501




In [75]:
import textwrap # Add this import
import subprocess, time, sys
from pyngrok import ngrok # Ensure ngrok is imported for restart logic

app_py_path = "/content/app/src/app.py"

with open(app_py_path, "r") as f:
    original_content = f.read()

# Define the new sidebar content directly
new_sidebar_content = textwrap.dedent('''
    with st.sidebar:
        st.markdown("""
        <div style="padding:20px 8px 8px">
          <div style="font-family:'Space Grotesk',sans-serif;font-size:20px;font-weight:700;
                      background:linear-gradient(135deg,#6C63FF,#3ECFCF);
                      -webkit-background-clip:text;-webkit-text-fill-color:transparent;
                      background-clip:text;margin-bottom:4px">🔍 AksaraDetect</div>
          <div style="font-size:11px;color:#8888AA;margin-bottom:20px">
            YOLOv8 · Aksara Ulu Rejang
          </div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown('<div style="font-size:11px;font-weight:600;letter-spacing:1px; text-transform:uppercase;color:#8888AA;margin-bottom:10px;padding:0 8px;">MENU NAVIGASI</div>', unsafe_allow_html=True)

        page = st.sidebar.radio(
            "", # Empty label to hide the default Streamlit radio label
            ["🏠 Home", "🚀 Detect", "📊 Analytics", "📜 History", "⚙️ Settings"], # Added icons
            key="main_nav",
        )

        st.markdown('<div style="height:1px;background:linear-gradient(90deg,transparent, rgba(108,99,255,0.4),transparent);margin:16px 0"></div>', unsafe_allow_html=True)

        # System Status
        st.markdown('<div style="font-size:11px;font-weight:600;letter-spacing:1px; text-transform:uppercase;color:#8888AA;margin-bottom:10px;padding:0 8px;">SYSTEM STATUS</div>', unsafe_allow_html=True)

        model_ok = os.path.isfile(config.MODEL_BEST_PATH)
        yaml_ok  = os.path.isfile(config.DATA_YAML)
        log      = utils.load_prediction_log()

        st.markdown(f"""
        <div style="display:flex;flex-direction:column;gap:8px">
          <div style="display:flex;justify-content:space-between;align-items:center;
                      background:rgba(255,255,255,0.03);border-radius:8px;padding:8px 12px">
            <span style="font-size:12px;color:#C0C0D8">YOLOv8 Model</span>
            <span style="font-size:11px;font-weight:600;
                         color:{'#4CAF50' if model_ok else '#F44336'}">
              {'● Ready' if model_ok else '○ Not trained'}
            </span>
          </div>
          <div style="display:flex;justify-content:space-between;align-items:center;
                      background:rgba(255,255,255,0.03);border-radius:8px;padding:8px 12px">
            <span style="font-size:12px;color:#C0C0D8">Dataset</span>
            <span style="font-size:11px;font-weight:600;
                         color:{'#4CAF50' if yaml_ok else '#F44336'}">
              {'● 253 kelas' if yaml_ok else '○ Not found'}
            </span>
          </div>
          <div style="display:flex;justify-content:space-between;align-items:center;
                      background:rgba(255,255,255,0.03);border-radius:8px;padding:8px 12px">
            <span style="font-size:12px;color:#C0C0D8">Deteksi Log</span>
            <span style="font-size:11px;font-weight:600;color:#6C63FF">{len(log)}</span>
          </div>
        </div>
        """, unsafe_allow_html=True)
    ''')

# Find the start and end of the current sidebar block in app.py
sideline_start = original_content.find('# ── Sidebar ───────────────────────────────────────────────────────────────────')
sideline_end   = original_content.find('# ── Route ─────────────────────────────────────────────────────────────────────')

if sideline_start != -1 and sideline_end != -1:
    # Replace the entire sidebar block
    new_app_content = original_content[:sideline_start] + new_sidebar_content + original_content[sideline_end:]
else:
    # Fallback if markers are not found, or handle error
    new_app_content = original_content # No change, log error or inform user
    print("Could not find sidebar markers in app.py. Manual replacement might be needed.")

# Update routing based on new labels with icons in a separate step or carefully within the string replace
# For simplicity, the routing logic `if 'Home' in page` etc. still works because I'm just adding emojis to the labels,
# but the string "Home" is still present within "🏠 Home". This is robust.

with open(app_py_path, "w") as f:
    f.write(new_app_content)

print("Sidebar in app.py updated for better UI layout and navigation!")

# Restart Streamlit after this change

# Kill proses lama
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Clear existing ngrok tunnels
for tunnel_obj in ngrok.get_tunnels():
    ngrok.disconnect(tunnel_obj.public_url)
    print(f"Closed old ngrok tunnel: {tunnel_obj.public_url}")

# Jalankan ulang Streamlit dengan `stderr` ke log file
log_file_path = "/content/app/src/streamlit_errors.log"
os.makedirs(os.path.dirname(log_file_path), exist_ok=True)

with open(log_file_path, "w") as log_file:
    proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "/content/app/src/app.py",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
            "--server.enableCORS",
            "false",
            "--server.enableXsrfProtection",
            "false",
        ],
        cwd="/content/app/src",
        stdout=log_file,
        stderr=log_file,
    )

time.sleep(10) # Beri waktu Streamlit untuk startup

tunnel = ngrok.connect(8501)
print("=" * 50)
print("AKSARADETECT SIAP (Navbar dan UI Sidebar diperbarui)!")
print("Buka:", tunnel.public_url)
print("=" * 50)


Sidebar in app.py updated for better UI layout and navigation!


Closed old ngrok tunnel: https://storage-evacuee-skeptic.ngrok-free.dev
AKSARADETECT SIAP (Navbar dan UI Sidebar diperbarui)!
Buka: https://storage-evacuee-skeptic.ngrok-free.dev


In [56]:
print('\nListing files in /content/app/src/_pages:')
!ls -l /content/app/src/_pages



Listing files in /content/app/src/_pages:
total 68
-rw-r--r-- 1 root root 5034 May 11 03:51 analytics.py
-rw-r--r-- 1 root root 8832 May 11 04:39 detect.py
-rw-r--r-- 1 root root 7463 May 11 03:51 history.py
-rw-r--r-- 1 root root 4672 May 11 03:51 home.py
-rw-r--r-- 1 root root   16 May 11 03:51 __init__.py
-rw-r--r-- 1 root root 3025 May 11 03:51 predict.py
drwxr-xr-x 2 root root 4096 May 11 04:40 __pycache__
-rw-r--r-- 1 root root 6119 May 11 04:54 settings.py
-rw-r--r-- 1 root root 8820 May 11 03:51 training.py


In [57]:
print('\nReading content of /content/app/src/_pages/home.py:')
!cat /content/app/src/_pages/home.py



Reading content of /content/app/src/_pages/home.py:
"""pages/home.py — Halaman beranda."""

import os, sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import streamlit as st
import config, utils
from components.ui import page_header, divider, stat_card


def render():
    st.markdown("""
    <div style="text-align:center;padding:48px 20px 32px">
      <div style="font-size:64px;margin-bottom:16px">🔍</div>
      <div style="font-family:'Space Grotesk',sans-serif;font-size:48px;font-weight:700;
                  background:linear-gradient(135deg,#6C63FF,#3ECFCF,#FF6584);
                  -webkit-background-clip:text;-webkit-text-fill-color:transparent;
                  background-clip:text;line-height:1.1;margin-bottom:12px">
        AksaraDetect
      </div>
      <div style="font-size:18px;color:#8888AA;max-width:540px;margin:0 auto 32px;line-height:1.6">
        Deteksi aksara Ulu Rejang secara otomatis menggunakan YOLOv8.
        253 kelas aksa

In [18]:
# Fix indentasi _draw_boxes di detect.py
with open("/content/app/src/_pages/detect.py", "r") as f:
    lines = f.readlines()

# Cari dan ganti baris yang bermasalah
new_lines = []
i = 0
while i < len(lines):
    line = lines[i]
    if "tostring_rgb" in line or "buffer_rgba" in line:
        # Ganti dengan versi yang benar
        indent = "    "
        new_lines.append(indent + "fig.canvas.draw()\n")
        new_lines.append(indent + "w_fig, h_fig = fig.canvas.get_width_height()\n")
        new_lines.append(indent + "buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_fig, w_fig, -1)[:, :, :3]\n")
    elif "w_fig, h_fig = fig.canvas.get_width_height()" in line:
        pass  # skip baris duplikat
    else:
        new_lines.append(line)
    i += 1

with open("/content/app/src/_pages/detect.py", "w") as f:
    f.writelines(new_lines)

print("Fixed!")

# Verifikasi
with open("/content/app/src/_pages/detect.py", "r") as f:
    content = f.read()

# Cek tidak ada error syntax
import ast
try:
    ast.parse(content)
    print("Syntax OK!")
except SyntaxError as e:
    print(f"Syntax error: {e}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/app/src/_pages/detect.py'

In [60]:
print('Listing files in /content/app/src/pages:')
!ls -l /content/app/src/pages

print('\nListing files in /content/app/src/_pages:')
!ls -l /content/app/src/_pages


Listing files in /content/app/src/pages:
ls: cannot access '/content/app/src/pages': No such file or directory

Listing files in /content/app/src/_pages:
total 68
-rw-r--r-- 1 root root 5034 May 11 03:51 analytics.py
-rw-r--r-- 1 root root 8761 May 11 04:57 detect.py
-rw-r--r-- 1 root root 7463 May 11 03:51 history.py
-rw-r--r-- 1 root root 4672 May 11 03:51 home.py
-rw-r--r-- 1 root root   16 May 11 03:51 __init__.py
-rw-r--r-- 1 root root 3025 May 11 03:51 predict.py
drwxr-xr-x 2 root root 4096 May 11 04:40 __pycache__
-rw-r--r-- 1 root root 6119 May 11 04:54 settings.py
-rw-r--r-- 1 root root 8820 May 11 03:51 training.py


In [70]:
# Fix - rename folder pages agar tidak dibaca otomatis Streamlit dan update app.py
import os, shutil, subprocess, time, sys

# Kill proses lama
subprocess.run([sys.executable, "-m", "streamlit", "run", "."], capture_output=True)
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Rename folder pages -> _pages (underscore = tidak dibaca otomatis)
if os.path.exists("/content/app/src/pages"):
    os.rename("/content/app/src/pages", "/content/app/src/_pages")
    print("Folder pages -> _pages")
elif not os.path.exists("/content/app/src/_pages"):
    print("Folder /content/app/src/pages tidak ditemukan dan /content/app/src/_pages juga tidak ada.")
else:
    print("Folder pages sudah di-rename menjadi _pages sebelumnya.")

# Move predict.py from src/ to src/_pages/ if it exists in src/ (this should only apply if predict.py is directly in src/ and not a page)
# Assuming predict.py is a page, it should be in _pages already.
# If it is src/predict.py and meant to be a page, it should be moved.
# Based on `ls -l /content/app/src/pages`, predict.py was a page, so it should be in _pages now.
# Let's check for /content/app/src/predict.py only.
if os.path.exists("/content/app/src/predict.py"):
    shutil.move("/content/app/src/predict.py", "/content/app/src/_pages/predict.py")
    print("Moved predict.py from src/ to _pages/!")

# Update all imports in app.py to use _pages
app_py_path = "/content/app/src/app.py"
with open(app_py_path, "r") as f:
    content = f.read()

# Only replace if 'pages.' exists to avoid unnecessary writes/modification loop
if "from pages." in content or "import pages." in content:
    # Using splitlines to avoid issues if 'pages.' appears in multiline strings
    new_content_lines = []
    for line in content.splitlines():
        new_line = line.replace("from pages.", "from _pages.")
        new_line = new_line.replace("import pages.", "import _pages.")
        new_content_lines.append(new_line)
    content = "\n".join(new_content_lines)

    with open(app_py_path, "w") as f:
        f.write(content)
    print("Imports updated in app.py!")
elif "from _pages." in content and "import _pages." in content:
    print("Imports in app.py already updated or no changes needed.")
else:
    print("No 'pages.' or '_pages.' imports found in app.py.")

# Clean up __pycache__ directories
for root, dirs, files in os.walk("/content/app/src"):
    if '__pycache__' in dirs:
        shutil.rmtree(os.path.join(root, '__pycache__'))
        print(f"Removed __pycache__ in {root}")

# Jalankan ulang Streamlit dengan `stderr` ke log file
log_file_path = "/content/app/src/streamlit_errors.log"
os.makedirs(os.path.dirname(log_file_path), exist_ok=True)

with open(log_file_path, "w") as log_file:
    proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "/content/app/src/app.py",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
            "--server.enableCORS",
            "false",
            "--server.enableXsrfProtection",
            "false",
        ],
        cwd="/content/app/src", # Jalankan dari src agar path relatif _pages tetap bekerja
        stdout=log_file,
        stderr=log_file,
    )

time.sleep(10)

from pyngrok import ngrok
# Tutup semua tunnel yang aktif sebelumnya untuk menghindari limit
for tunnel_obj in ngrok.get_tunnels():
    ngrok.disconnect(tunnel_obj.public_url)
    print(f"Closed old ngrok tunnel: {tunnel_obj.public_url}")

tunnel = ngrok.connect(8501)
print("=" * 50)
print("AKSARADETECT SIAP (Setelah perbaikan app.py imports)!")
print("Buka:", tunnel.public_url)
print("=" * 50)


Folder pages sudah di-rename menjadi _pages sebelumnya.
No 'pages.' or '_pages.' imports found in app.py.


Closed old ngrok tunnel: https://storage-evacuee-skeptic.ngrok-free.dev
AKSARADETECT SIAP (Setelah perbaikan app.py imports)!
Buka: https://storage-evacuee-skeptic.ngrok-free.dev


In [58]:
# Fix tostring_rgb -> buffer_rgba dalam _draw_boxes di detect.py
# Perbaikan ini sudah disiapkan sebelumnya oleh Kiro.

import numpy as np

file_path = "/content/app/src/_pages/detect.py"

with open(file_path, "r") as f:
    lines = f.readlines()

new_lines = []
i = 0
while i < len(lines):
    line = lines[i]
    if "buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)" in line:
        # Original line (line 66 of _pages/detect.py in the previous trace)
        # Replace with the correct logic including fig.canvas.draw() and buffer_rgba()
        indent = line.split("buf =")[0] # Preserve indentation
        new_lines.append(indent + "fig.canvas.draw()\n")
        new_lines.append(indent + "w_fig, h_fig = fig.canvas.get_width_height()\n")
        new_lines.append(indent + "buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_fig, w_fig, -1)[:, :, :3]\n")
    elif "buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h, w, -1)[:, :, :3]" in line or \
         "w_fig, h_fig = fig.canvas.get_width_height()" in line or \
         "fig.canvas.draw()" in line:
        # Skip these lines if they were added by previous failed attempts to avoid duplicates
        pass
    else:
        new_lines.append(line)
    i += 1

with open(file_path, "w") as f:
    f.writelines(new_lines)

print("Fixed 'tostring_rgb' issue in detect.py!")

# Verifikasi syntax setelah perubahan
import ast
with open(file_path, "r") as f:
    content = f.read()
try:
    ast.parse(content)
    print("Syntax OK!")
except SyntaxError as e:
    print(f"Syntax error found after fix: {e}")


Fixed 'tostring_rgb' issue in detect.py!
Syntax OK!


In [59]:
print('\nReading content of /content/app/src/utils.py:')
!cat /content/app/src/utils.py



Reading content of /content/app/src/utils.py:
"""
utils.py — Fungsi pembantu: label map, logging prediksi, visualisasi.
"""

import os
import json
import datetime
import yaml
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import config

PALETTE = {
    "primary":   "#6C63FF",
    "secondary": "#3ECFCF",
    "accent":    "#FF6584",
    "dark":      "#1A1A2E",
    "surface":   "#16213E",
    "text":      "#E0E0E0",
}


# ── Direktori ─────────────────────────────────────────────────────────────────

def ensure_dirs():
    for d in [config.MODEL_DIR, config.RESULT_DIR, config.LOG_DIR, config.RUNS_DIR]:
        os.makedirs(d, exist_ok=True)


# ── Label Map ─────────────────────────────────────────────────────────────────

def load_label_map_from_yaml(yaml_path: str = config.DATA_YAML) -> dict:
    """Baca nama kelas dari data.yaml → {index: nama}."""
    with open(yaml_path, "r") as f:

In [73]:
import os, shutil, subprocess, time, sys
from pyngrok import ngrok

css_content = '''
/* ===================================================================
   AksaraDetect — Global Theme Enhancements
   Modern dark glassmorphism + vibrant accent colors
=================================================================== */

/* -- Google Fonts -- */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=Space+Grotesk:wght@400;500;600;700&display=swap');

/* -- Root Variables -- */
:root {
  --primary:      #6C63FF;
  --primary-dark: #5A52E0;
  --secondary:    #3ECFCF;
  --accent:       #FF6584;
  --success:      #4CAF50;
  --warning:      #FF9800;
  --danger:       #F44336;
  --bg-base:      #0F0F1A;
  --bg-surface:   #1A1A2E;
  --bg-card:      #16213E;
  --bg-glass:     rgba(255,255,255,0.04);
  --border:       rgba(108,99,255,0.2);
  --border-hover: rgba(108,99,255,0.5);
  --text-primary: #F0F0FF;
  --text-muted:   #8888AA;
  --radius-sm:    8px;
  --radius-md:    14px;
  --radius-lg:    20px;
  --shadow:       0 8px 32px rgba(0,0,0,0.4);
  --shadow-glow:  0 0 24px rgba(108,99,255,0.25);

  /* New variables for consistency and control */
  --font-family-header: 'Space Grotesk', sans-serif;
  --font-family-body:   'Inter', sans-serif;
  --font-weight-bold:   700;
  --spacing-unit:       16px;
  --transition-speed:   0.3s;
}

/* -- Base -- */
html, body, [data-testid="stAppViewContainer"] {
  background: var(--bg-base) !important;
  font-family: var(--font-family-body) !important;
  color: var(--text-primary) !important;
  line-height: 1.6;
}

[data-testid="stSidebar"] {
  background: var(--bg-surface) !important;
  border-right: 1px solid var(--border) !important;
  box-shadow: var(--shadow);
}

/* -- Hide default Streamlit chrome -- */
#MainMenu, footer, header { visibility: hidden; }
[data-testid="stDecoration"] { display: none; }

/* -- Scrollbar -- */
::-webkit-scrollbar { width: 8px; height: 8px; }
::-webkit-scrollbar-track { background: var(--bg-surface); border-radius: 4px; }
::-webkit-scrollbar-thumb { background: var(--primary); border-radius: 4px; }
::-webkit-scrollbar-thumb:hover { background: var(--primary-dark); }

/* -- Headings -- */
h1, h2, h3, h4 {
  font-family: var(--font-family-header) !important;
  color: var(--text-primary) !important;
  margin-bottom: calc(var(--spacing-unit) / 2);
  line-height: 1.2;
}

/* -- Streamlit widgets -- */
[data-testid="stSelectbox"] > div,
[data-testid="stRadio"] > div,
[data-testid="stTextInput"] > div > div,
[data-testid="stTextArea"] > div > div,
[data-testid="stNumberInput"] > div > div {
  background: var(--bg-glass) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius-sm) !important;
  transition: border-color var(--transition-speed), box-shadow var(--transition-speed);
}
[data-testid="stSelectbox"] > div:focus-within,
[data-testid="stRadio"] > div:focus-within,
[data-testid="stTextInput"] > div > div:focus-within,
[data-testid="stTextArea"] > div > div:focus-within,
[data-testid="stNumberInput"] > div > div:focus-within {
  border-color: var(--primary) !important;
  box-shadow: 0 0 0 2px rgba(108,99,255,0.3) !important;
}

/* File Uploader */
[data-testid="stFileUploadDropzone"] {
  min-height: 150px;
  border-radius: var(--radius-md) !important;
  border: 2px dashed var(--border) !important;
  background: var(--bg-glass) !important;
  transition: border-color var(--transition-speed), background var(--transition-speed);
}
[data-testid="stFileUploadDropzone"]:hover {
  border-color: var(--primary) !important;
  background: rgba(108,99,255,0.05) !important;
}

/* Custom Card Component (if used with st.markdown) */
.vc-card {
  background: var(--bg-card) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius-md) !important;
  padding: var(--spacing-unit) !important;
  transition: transform var(--transition-speed), box-shadow var(--transition-speed);
}
.vc-card:hover {
  transform: translateY(-3px);
  box-shadow: var(--shadow-glow);
}
.vc-card-title {
  font-family: var(--font-family-header);
  font-weight: var(--font-weight-bold);
  color: var(--text-primary);
  margin-bottom: calc(var(--spacing-unit) / 4);
}

/* -- Buttons -- */
.stButton > button {
  background: linear-gradient(135deg, var(--primary), var(--primary-dark)) !important;
  color: white !important;
  border: none !important;
  border-radius: var(--radius-sm) !important;
  font-weight: var(--font-weight-bold) !important;
  letter-spacing: 0.5px !important;
  transition: all var(--transition-speed) ease !important;
  box-shadow: 0 4px 15px rgba(108,99,255,0.3) !important;
  padding: 10px 20px;
}
.stButton > button:hover {
  transform: translateY(-2px) !important;
  box-shadow: 0 8px 25px rgba(108,99,255,0.5) !important;
  background: linear-gradient(135deg, var(--primary-dark), var(--primary)) !important; /* Subtle gradient shift */
}

/* -- Tabs -- */
[data-testid="stTabs"] [role="tab"] {
  color: var(--text-muted) !important;
  font-weight: 500 !important;
  border-radius: var(--radius-sm) var(--radius-sm) 0 0 !important;
  transition: color var(--transition-speed), border-bottom-color var(--transition-speed);
  padding: 10px 15px;
}
[data-testid="stTabs"] [role="tab"][aria-selected="true"] {
  color: var(--primary) !important;
  border-bottom: 3px solid var(--primary) !important; /* Slightly thicker active tab border */
}
[data-testid="stTabs"] > div:first-child {
  border-bottom: 1px solid var(--border); /* Separator for tabs */
}

/* -- Expander -- */
[data-testid="stExpander"] {
  background: var(--bg-glass) !important;
  border: 1px solid var(--border) !important;
  border-radius: var(--radius-md) !important;
  padding: 1px;
  transition: border-color var(--transition-speed);
}
[data-testid="stExpander"]:hover {
  border-color: var(--primary);
}

/* -- DataFrame -- */
[data-testid="stDataFrame"] {
  border: 1px solid var(--border) !important;
  border-radius: var(--radius-md) !important;
  overflow: hidden;
  box-shadow: var(--shadow);
}

/* -- Progress bar -- */
[data-testid="stProgress"] > div > div {
  background: linear-gradient(90deg, var(--primary), var(--secondary)) !important;
  border-radius: 4px !important;
  height: 8px; /* Slightly thicker progress bar */
}

/* -- Alerts -- */
[data-testid="stAlert"] {
  border-radius: var(--radius-md) !important;
  border: none !important;
  background-color: var(--bg-card) !important;
  box-shadow: var(--shadow);
}

/* -- Sidebar nav items -- */
[data-testid="stSidebarNav"] a {
  border-radius: var(--radius-sm) !important;
  transition: background var(--transition-speed), color var(--transition-speed);
  padding: 10px 15px;
  color: var(--text-muted) !important;
}
[data-testid="stSidebarNav"] a:hover {
  background: var(--bg-glass) !important;
  color: var(--text-primary) !important;
}
[data-testid="stSidebarNav"] a.active {
    background: var(--primary) !important; /* Highlight active page */
    color: white !important;
    font-weight: var(--font-weight-bold);
}

/* -- Metric Cards -- */
[data-testid="stMetric"] {
    background-color: var(--bg-card);
    border: 1px solid var(--border);
    border-radius: var(--radius-md);
    padding: var(--spacing-unit);
    box-shadow: var(--shadow);
    transition: transform var(--transition-speed), box-shadow var(--transition-speed);
}
[data-testid="stMetric"]:hover {
    transform: translateY(-3px);
    box-shadow: var(--shadow-glow);
}
[data-testid="stMetricLabel"] div {
    font-family: var(--font-family-header);
    color: var(--text-muted);
    font-size: 0.9em;
}
[data-testid="stMetricValue"] {
    font-family: var(--font-family-header);
    font-weight: var(--font-weight-bold);
    color: var(--primary);
    font-size: 1.8em;
}
[data-testid="stMetricDelta"] div {
    color: var(--secondary);
    font-size: 0.8em;
}

/* -- General text and markdown -- */
p {
  margin-bottom: calc(var(--spacing-unit) / 2);
}

/* -- Custom Markdown for better code blocks -- */
code {
  background-color: rgba(255,255,255,0.08) !important;
  border-radius: 4px;
  padding: 2px 4px;
  font-family: 'Fira Code', monospace;
  color: #3ECFCF;
}
pre code {
  background-color: rgba(255,255,255,0.05) !important;
  border: 1px solid var(--border);
  display: block;
  padding: var(--spacing-unit);
  border-radius: var(--radius-md);
  overflow-x: auto;
}

'''

# Tulis konten CSS yang baru ke theme.css
css_file_path = "/content/app/src/styles/theme.css"
with open(css_file_path, "w") as f:
    f.write(css_content)

print("CSS theme.css updated successfully!")

# Kill proses lama Streamlit
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Clear existing ngrok tunnels
for tunnel_obj in ngrok.get_tunnels():
    ngrok.disconnect(tunnel_obj.public_url)
    print(f"Closed old ngrok tunnel: {tunnel_obj.public_url}")

# Jalankan ulang Streamlit dengan `stderr` ke log file
log_file_path = "/content/app/src/streamlit_errors.log"
os.makedirs(os.path.dirname(log_file_path), exist_ok=True)

with open(log_file_path, "w") as log_file:
    proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "/content/app/src/app.py",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
            "--server.enableCORS",
            "false",
            "--server.enableXsrfProtection",
            "false",
        ],
        cwd="/content/app/src",
        stdout=log_file,
        stderr=log_file,
    )

time.sleep(10) # Beri waktu Streamlit untuk startup

tunnel = ngrok.connect(8501)
print("=" * 50)
print("AKSARADETECT SIAP (UI diperbarui)!")
print("Buka:", tunnel.public_url)
print("=" * 50)


CSS theme.css updated successfully!


Closed old ngrok tunnel: https://storage-evacuee-skeptic.ngrok-free.dev
AKSARADETECT SIAP (UI diperbarui)!
Buka: https://storage-evacuee-skeptic.ngrok-free.dev
